# Notebook 02 — AgentCore Memory

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS — Workshop 2

---

What changes when an agent can remember context across invocations?

This notebook answers that question with working code. By the end, you will have
seen how the conversational-context-manager uses AgentCore Memory to maintain
session state — and you will understand why memory is session-scoped, not permanent.

In [ ]:
# WS2 notebook setup — installs dependencies into THIS kernel.
# uv manages the WS2 virtual environment; this cell installs it
# into the running kernel so imports work without manual setup.
import sys, subprocess, os
from pathlib import Path

# Find the use-case-applications root (3 levels up from phase-1-referral/)
ws2_root = Path(os.getcwd()).parents[1]
venv_python = ws2_root / '.venv' / 'bin' / 'python'

if venv_python.exists() and str(venv_python) != sys.executable:
    # venv exists but we're not running inside it — install into current kernel
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--disable-pip-version-check',
         'rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0',
         'pydantic>=2.0.0'],
        cwd='/tmp'
    )
elif not venv_python.exists():
    # venv not yet created — run uv sync first
    subprocess.check_call(
        ['uv', 'sync', '--all-groups', '--quiet'],
        cwd=str(ws2_root)
    )
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--disable-pip-version-check',
         'rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0',
         'pydantic>=2.0.0'],
        cwd='/tmp'
    )

print('WS2 dependencies ready.')


## Key terms for this notebook

| Term | What it is |
|------|------------|
| **AgentCore Memory** | An AWS service that provides session-scoped key-value storage for agents. Data persists within a session but is cleared when the session ends. No permanent user state is retained. |
| **Session scope** | The boundary within which memory is valid. A session starts when a user begins an interaction and ends when they close the conversation or a timeout expires. Memory does not carry across sessions. |
| **Conversational-context-manager** | The Phase 2 agent responsible for maintaining multi-turn context. It stores prior query results and user references so that follow-up questions like "Of those, which..." resolve correctly. |
| **Context resolution** | The process of resolving anaphoric references ("those", "the first one", "that client") by looking up what the prior turn returned. Without memory, these references are unresolvable. |
| **Template selection with context** | When memory contains prior results, the template selector can use that context to narrow which SPARQL template applies — e.g., adding a filter for entities already returned. |

## Memory makes follow-up questions possible

Phase 1 agents are stateless. Each invocation receives a question, selects a
SPARQL template, executes it, and returns the result. The agent does not know what
was asked before, what was returned before, or who the user was referring to when
they said "those." This is acceptable for the Consumer Banker workflow, where each
question is self-contained: "Which customers have no wealth advisor?" stands alone.
But a Wealth Advisor's workflow is conversational. They ask "Show me clients with
engagement decay" and then follow up with "Of those, which have AUM above 2M?" The
second question is meaningless without the first — "those" has no referent unless
the agent remembers what it returned.

The conversational-context-manager solves this by storing each turn's result in
AgentCore Memory under the current session ID. When a new question arrives, the
agent first checks memory for prior context. If context exists, it resolves
anaphoric references — "those" becomes the list of client URIs from the previous
result — and passes the resolved context to the template selector. The selector
can then choose a template that filters by those specific URIs, producing a query
that answers the follow-up correctly.

Memory is deliberately session-scoped. When the session ends — whether by explicit
close, timeout, or user navigation away — all stored context is cleared. There is
no permanent record of what the advisor asked or what was returned. This is a
privacy and compliance decision: storing conversation history permanently would
create a data retention obligation under GDPR and CCPA, and would require the same
governance controls as any other customer data store. Session-scoped memory avoids
that entirely — it exists only as long as the conversation is active, and it
disappears when the conversation ends.

This design also means that the conversational-context-manager is not a knowledge
base. It does not learn from past sessions, it does not build user profiles, and
it does not accumulate preferences over time. Each session starts fresh. The agent
is a short-term memory buffer, not a long-term store — and that constraint is what
makes it safe to deploy in a regulated environment without additional data
governance overhead.

In [ ]:
import sys
import os
import json
import uuid

# Workshop 1's shared helpers.
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

from pathlib import Path

SPEC_DIR = "../../spec/04-aws-agent-registry"

print("Setup complete.")
print("This notebook simulates AgentCore Memory locally.")
print("Production uses the AWS AgentCore Memory service.")

In [ ]:
# Build cell 1 — Simulate AgentCore Memory as a session-scoped store.
#
# In production, this is an AWS service call. Here we implement the
# same interface locally to demonstrate the session-scoped contract.

class SessionMemory:
    """Local simulation of AgentCore Memory.
    
    Session-scoped: data exists only within a session.
    No persistence across sessions.
    """
    def __init__(self):
        self._sessions = {}  # session_id → {key: value}
    
    def put(self, session_id, key, value):
        if session_id not in self._sessions:
            self._sessions[session_id] = {}
        self._sessions[session_id][key] = value
    
    def get(self, session_id, key, default=None):
        return self._sessions.get(session_id, {}).get(key, default)
    
    def end_session(self, session_id):
        """Clear all memory for a session."""
        self._sessions.pop(session_id, None)
    
    def active_sessions(self):
        return list(self._sessions.keys())

memory = SessionMemory()
print("SessionMemory initialized.")
print("Interface: put(session_id, key, value), get(session_id, key), end_session(session_id)")

In [ ]:
# Build cell 2 — Simulate a multi-turn conversation with memory.
#
# Turn 1: "Show me clients with engagement decay"
# Turn 2: "Of those, which have AUM above 2M?"
#
# The second turn resolves "those" from memory.

session_id = str(uuid.uuid4())
print(f"Session started: {session_id[:8]}...")
print()

# Turn 1 — simulate query result
turn_1_question = "Show me clients with engagement decay"
turn_1_results = [
    {"uri": "atlas:client-rachel-kim", "name": "Rachel Kim", "aum": 3200000},
    {"uri": "atlas:client-james-chen", "name": "James Chen", "aum": 1800000},
    {"uri": "atlas:client-sarah-patel", "name": "Sarah Patel", "aum": 4500000},
]

# Store in memory
memory.put(session_id, "last_results", turn_1_results)
memory.put(session_id, "last_question", turn_1_question)

print(f"Turn 1: \"{turn_1_question}\"")
print(f"Results stored in memory: {len(turn_1_results)} clients")
print()

# Turn 2 — resolve "those" from memory
turn_2_question = "Of those, which have AUM above 2M?"
prior_results = memory.get(session_id, "last_results", [])

# Context resolution: "those" → prior results
filtered = [c for c in prior_results if c["aum"] > 2_000_000]

print(f"Turn 2: \"{turn_2_question}\"")
print(f"Resolved 'those' from memory: {len(prior_results)} clients")
print(f"After AUM filter: {len(filtered)} clients")
print()
for c in filtered:
    print(f"  {c['name']:<20} AUM: ${c['aum']:,.0f}")

In [ ]:
# Build cell 3 — Show how prior context influences template selection.
#
# Without memory: "which have AUM above 2M?" selects a broad AUM query.
# With memory: the same question selects a filtered query scoped to
# the URIs from the prior turn.

TEMPLATES = {
    "aum_filter_broad": "SELECT ?client WHERE { ?client atlas:aum ?aum . FILTER(?aum > $threshold) }",
    "aum_filter_scoped": "SELECT ?client WHERE { VALUES ?client { $uris } ?client atlas:aum ?aum . FILTER(?aum > $threshold) }",
}

def select_template(question, session_id, memory_store):
    """Select template based on whether memory has prior context."""
    prior = memory_store.get(session_id, "last_results")
    has_anaphora = any(w in question.lower() for w in ["those", "them", "these"])
    
    if prior and has_anaphora:
        uris = " ".join(f"<{c['uri']}>" for c in prior)
        return "aum_filter_scoped", {"uris": uris, "threshold": "2000000"}
    else:
        return "aum_filter_broad", {"threshold": "2000000"}

# With memory (session active)
template, params = select_template(turn_2_question, session_id, memory)
print(f"With memory active:")
print(f"  Template: {template}")
print(f"  Params:   {params}")
print()

# Without memory (new session)
new_session = str(uuid.uuid4())
template2, params2 = select_template(turn_2_question, new_session, memory)
print(f"Without memory (new session):")
print(f"  Template: {template2}")
print(f"  Params:   {params2}")

## Verification

Two properties must hold: memory is session-scoped (cleared when the session ends),
and context does not leak between sessions. If either fails, the system would either
retain data beyond its intended lifetime or allow one user's context to influence
another user's results.

In [ ]:
# Verification cell 1 — Memory is session-scoped (clears at end).

print("Verifying session-scoped memory behavior...")
print()

# Create a session and store data
test_session = str(uuid.uuid4())
memory.put(test_session, "test_key", "test_value")

# Verify data exists during session
before_end = memory.get(test_session, "test_key")
print(f"Before end_session: memory.get('test_key') = {before_end}")

# End the session
memory.end_session(test_session)

# Verify data is cleared
after_end = memory.get(test_session, "test_key")
print(f"After end_session:  memory.get('test_key') = {after_end}")
print()

if after_end is not None:
    print("VERIFICATION FAILED: Memory persists after session end.")
    print("The end_session() method must clear all data for the session.")
    print("Check the SessionMemory.end_session() implementation.")

assert before_end == "test_value", "Memory should store values during active session."
assert after_end is None, (
    "Memory must be cleared after end_session(). "
    "Session-scoped means no data persists beyond the session lifetime."
)

print("[PASS] Memory is session-scoped: data cleared after end_session().")

In [ ]:
# Verification cell 2 — Context does not leak between sessions.

print("Verifying session isolation...")
print()

session_a = str(uuid.uuid4())
session_b = str(uuid.uuid4())

# Store different data in each session
memory.put(session_a, "clients", ["Rachel Kim", "James Chen"])
memory.put(session_b, "clients", ["Sarah Patel"])

# Verify isolation
a_clients = memory.get(session_a, "clients")
b_clients = memory.get(session_b, "clients")

print(f"Session A clients: {a_clients}")
print(f"Session B clients: {b_clients}")
print()

# End session A — session B should be unaffected
memory.end_session(session_a)
b_after = memory.get(session_b, "clients")

print(f"After ending session A:")
print(f"  Session A clients: {memory.get(session_a, 'clients')}")
print(f"  Session B clients: {b_after}")
print()

if b_after != ["Sarah Patel"]:
    print("VERIFICATION FAILED: Ending session A affected session B.")
    print("Sessions must be fully isolated. Check that end_session()")
    print("only removes the specified session's data.")

assert a_clients != b_clients, "Sessions must contain different data."
assert b_after == ["Sarah Patel"], (
    "Ending one session must not affect another. "
    "Session isolation is required for multi-user safety."
)

# Cleanup
memory.end_session(session_b)

print("[PASS] Sessions are fully isolated. No context leakage between sessions.")

## What just changed

You have seen how AgentCore Memory enables multi-turn conversations by storing
prior results within a session. The conversational-context-manager resolves
anaphoric references ("those", "them") by looking up what the previous turn
returned, then passes that context to the template selector for scoped queries.

Memory is session-scoped by design: it clears when the session ends, sessions are
fully isolated from each other, and no permanent user state is retained. This
avoids data retention obligations while enabling the conversational workflow that
a Wealth Advisor needs.

The next notebook shows how the same GraphQL backbone serves a structurally
different application — the Wealth UI — by rendering different fragments and
capabilities for the wealth-advisor persona.